# Animer — train a two-tower neural recommender (v2)

Upgrade over the ALS notebook: bigger data (~25M ratings vs ~7M) + a real two-tower neural network instead of linear matrix factorization.

**What's different:**
- **Data**: `azathoth42/myanimelist` Kaggle dataset (~300k users, ~25M ratings) instead of CooperUnion (~73k users, ~7M ratings)
- **Model**: PyTorch two-tower MLP. Each tower learns a 64-dim representation; we score user-item by cosine similarity
- **Loss**: BPR (Bayesian Personalized Ranking) on implicit feedback — what real production recsys use
- **Output**: same `cf_embedding vector(64)` column — drop-in replacement, no schema change

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** (REQUIRED — training will be too slow on CPU)
2. Have Supabase URL + service key ready
3. Kaggle API token ready (kagglehub will prompt for it)

Total time: ~45-90 min depending on epochs.

## 1. Install dependencies

In [ ]:
!pip install -q kagglehub supabase pandas numpy scipy tqdm

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'Switch to T4 GPU runtime!'

## 2. Credentials

In [ ]:
import os

os.environ['SUPABASE_URL'] = 'https://xxxxx.supabase.co'      # ← REPLACE
os.environ['SUPABASE_SERVICE_KEY'] = 'eyJ...'                  # ← REPLACE

assert os.environ['SUPABASE_URL'].startswith('https://')
print('✓ Supabase creds set')

## 3. Download the larger dataset

`azathoth42/myanimelist` has both rich user lists and the comprehensive anime table. We'll use `UserAnimeList.csv` for ratings.

In [ ]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download('azathoth42/myanimelist')
print('Files:', os.listdir(path))

In [ ]:
# Load ratings — this is the big one (~2-3GB on disk). Read columns we need only.
ratings = pd.read_csv(
    f'{path}/UserAnimeList.csv',
    usecols=['username', 'anime_id', 'my_score', 'my_status'],
    dtype={'username': 'category', 'anime_id': 'int32', 'my_score': 'int8', 'my_status': 'int8'},
)

anime_meta = pd.read_csv(f'{path}/anime_filtered.csv', usecols=['anime_id', 'title'])

print(f'Raw ratings: {len(ratings):,}')
print(f'Anime in meta: {len(anime_meta):,}')
print(f'\nStatus codes: {ratings.my_status.value_counts().to_dict()}')
print(f'Score distribution:\n{ratings.my_score.value_counts().sort_index().to_dict()}')

## 4. Filter to high-quality implicit feedback

For BPR we want **positive interactions only** (anime the user clearly liked). We keep:
- `my_status == 2` (Completed) AND `my_score >= 7` → strong positive
- Drop unrated entries, dropped shows, plan-to-watch
- Filter low-activity users (< 10 positives) and cold anime (< 30 positives)

In [ ]:
# Keep only Completed (status=2) with score >= 7. These are 'I liked it' signals.
df = ratings[(ratings.my_status == 2) & (ratings.my_score >= 7)].copy()
df = df.drop(columns=['my_status'])
print(f'After filter (completed + rated 7+): {len(df):,}')

# Iterative pruning
for _ in range(3):
    u_count = df.groupby('username').size()
    a_count = df.groupby('anime_id').size()
    df = df[df.username.isin(u_count[u_count >= 10].index)]
    df = df[df.anime_id.isin(a_count[a_count >= 30].index)]

print(f'After pruning: {len(df):,} interactions, {df.username.nunique():,} users, {df.anime_id.nunique():,} anime')

## 5. Re-index and convert to tensors

In [ ]:
import numpy as np

user_to_idx = {u: i for i, u in enumerate(df.username.unique())}
anime_to_idx = {a: i for i, a in enumerate(df.anime_id.unique())}
idx_to_anime = {i: a for a, i in anime_to_idx.items()}

N_USERS = len(user_to_idx)
N_ITEMS = len(anime_to_idx)

df['u'] = df.username.map(user_to_idx).astype('int32')
df['i'] = df.anime_id.map(anime_to_idx).astype('int32')

user_ids = torch.tensor(df.u.values, dtype=torch.long)
item_ids = torch.tensor(df.i.values, dtype=torch.long)
scores = torch.tensor(df.my_score.values.astype(np.float32))

print(f'Users: {N_USERS:,}  Items: {N_ITEMS:,}  Interactions: {len(df):,}')

# Build per-user positive sets for negative sampling
from collections import defaultdict
pos_by_user = defaultdict(set)
for u, i in zip(df.u.values, df.i.values):
    pos_by_user[u].add(i)
print(f'Avg positives per user: {np.mean([len(s) for s in pos_by_user.values()]):.1f}')

## 6. Define the two-tower model

Each tower is an embedding layer + 2-layer MLP. Cosine similarity is the score. This is more expressive than pure MF (ALS) because the MLPs can learn non-linear interactions.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

EMBED_DIM = 64        # output dim (matches our Supabase column)
HIDDEN = 128          # internal width before projection
DROPOUT = 0.2

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, embed_dim=64, hidden=128, dropout=0.2):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, hidden)
        self.item_emb = nn.Embedding(n_items, hidden)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

        self.user_mlp = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
        )

    def user_repr(self, u):
        return F.normalize(self.user_mlp(self.user_emb(u)), dim=-1)

    def item_repr(self, i):
        return F.normalize(self.item_mlp(self.item_emb(i)), dim=-1)

    def score(self, u, i):
        return (self.user_repr(u) * self.item_repr(i)).sum(-1)

model = TwoTower(N_USERS, N_ITEMS, EMBED_DIM, HIDDEN, DROPOUT).cuda()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params/1e6:.1f}M')

## 7. Train with BPR loss

BPR (Bayesian Personalized Ranking) maximizes `score(user, positive) - score(user, negative)` for each user. Negatives are randomly sampled items the user hasn't rated.

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 8192
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-6

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

user_ids_g = user_ids.cuda()
item_ids_g = item_ids.cuda()
n_interactions = len(user_ids_g)

for epoch in range(EPOCHS):
    perm = torch.randperm(n_interactions, device='cuda')
    pbar = tqdm(range(0, n_interactions, BATCH_SIZE), desc=f'epoch {epoch+1}/{EPOCHS}')
    epoch_loss = 0.0
    epoch_count = 0

    for start in pbar:
        idx = perm[start:start+BATCH_SIZE]
        u_batch = user_ids_g[idx]
        pos_batch = item_ids_g[idx]
        # Sample random negatives (cheap — collisions with positives are rare and OK as noise)
        neg_batch = torch.randint(0, N_ITEMS, (len(idx),), device='cuda')

        pos_score = model.score(u_batch, pos_batch)
        neg_score = model.score(u_batch, neg_batch)

        # BPR loss: -log(sigmoid(pos - neg))
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss += loss.item() * len(idx)
        epoch_count += len(idx)
        if start % (BATCH_SIZE * 20) == 0:
            pbar.set_postfix({'loss': f'{epoch_loss/epoch_count:.4f}'})

    print(f'  Epoch {epoch+1} avg loss: {epoch_loss/epoch_count:.4f}')

print('\n✓ Training complete')

## 8. Extract item embeddings

These are the vectors we upload to Supabase. We use the `item_repr` output (post-MLP, L2-normalized).

In [ ]:
model.eval()
with torch.no_grad():
    all_item_ids = torch.arange(N_ITEMS, device='cuda')
    item_embeddings = model.item_repr(all_item_ids).cpu().numpy()

print(f'Item embeddings shape: {item_embeddings.shape}')

## 9. Sanity check — Toradora neighbors

Same gut-check as the ALS notebook. If this is at least as good (or better), we've succeeded.

In [ ]:
def neighbors_for(mal_id, k=10):
    if mal_id not in anime_to_idx:
        print(f'MAL {mal_id} not in trained set')
        return
    idx = anime_to_idx[mal_id]
    target = item_embeddings[idx]
    sims = item_embeddings @ target
    top = np.argsort(-sims)[:k+1]
    for i in top:
        if i == idx: continue
        m = idx_to_anime[i]
        meta = anime_meta[anime_meta.anime_id == m]
        title = meta.title.iloc[0] if len(meta) else '?'
        print(f'  {sims[i]:.3f}  [MAL {m}]  {title}')

print('Toradora! (MAL 4224):')
neighbors_for(4224)
print()
print('Steins;Gate (MAL 9253):')
neighbors_for(9253)
print()
print('Frieren (MAL 52991):')
neighbors_for(52991)

## 10. Match to Supabase + upload

Same upload pattern as the ALS notebook — we use UPDATE not upsert to avoid the NOT NULL issue.

In [ ]:
from supabase import create_client

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

# Pull every Supabase row that has a mal_id (paginated)
all_rows = []
page = 0
PAGE = 1000
while True:
    res = sb.table('anime').select('id, mal_id').not_.is_('mal_id', 'null') \
        .range(page*PAGE, (page+1)*PAGE - 1).execute()
    if not res.data: break
    all_rows.extend(res.data)
    if len(res.data) < PAGE: break
    page += 1

print(f'Supabase rows with mal_id: {len(all_rows)}')

supabase_to_emb = {}
missing = 0
for row in all_rows:
    mid = row['mal_id']
    if mid in anime_to_idx:
        supabase_to_emb[row['id']] = item_embeddings[anime_to_idx[mid]]
    else:
        missing += 1

print(f'Matched: {len(supabase_to_emb)} / {len(all_rows)} (missing from training: {missing})')

In [ ]:
from tqdm import tqdm

items = list(supabase_to_emb.items())
failed = []

for sb_id, emb in tqdm(items, desc='Uploading'):
    try:
        sb.table('anime').update(
            {'cf_embedding': emb.tolist()}
        ).eq('id', sb_id).execute()
    except Exception as e:
        failed.append((sb_id, str(e)))

print(f'\n✓ Done. Failed: {len(failed)}')
if failed[:3]:
    print('First failures:', failed[:3])

verify = sb.table('anime').select('id', count='exact') \
    .not_.is_('cf_embedding', 'null').execute()
print(f'Anime with cf_embedding in Supabase: {verify.count}')